In [ ]:
BEAM_WIDTH = 2**26
WORLD_SIZE = 2
START_PUZZLE_ID = 0
PUZZLE_COUNT = 1
DEPTH_LIMIT = 16
RUN_TIMEOUT_SEC = 600
GITHUB_REPO_URL = "https://github.com/TryDotAtwo/MultiGPUBeamSearch.git"
GITHUB_BRANCH = "main"
SOURCE_ARCHIVE_DATASET = "cayley-beam-solver-source-multigpu"
SOURCE_ARCHIVE_NAME = "beam_solver_source.tar.gz"
ENABLE_DEBUG = True
ENABLE_DEPTH_LOGS = True
ENABLE_DEBUG_LOGS = True
DEBUG_STREAM_TIMING = True
DEBUG_INFERENCE_TRACE = True
DEBUG_PATH_TRACE = True
DEBUG_FINAL_VALIDATE = True
DEBUG_FINAL_EXCHANGE_TRACE = True
DEBUG_FINAL_HISTOGRAM_TRACE = True
DEBUG_STREAM4_HISTOGRAM_TRACE = True
DEBUG_PIPELINE_STATS = True
DEPTH_LOG_EVERY = 1
PUZZLE_LOG_EVERY = 1
HISTORY_MODE = "ram"
HISTORY_SLOT_COUNT = 3
HISTORY_WORKERS = 1
STOP_ON_FAILURE = False
LIVE_LOG_RANKS = "all"  # "all", "none", or a list like [0, 1]

RUNTIME_CONFIG_MODE = "manual"
SHARD_BUFFER_COUNT = 2
STREAM3_RING_SLOTS = 1
SHARD_COUNT = 4
STREAM4_BATCH_ALIGNMENT = 1024
SHARD_CAPACITY_SCALE_PPM = 1250000
STREAM4_BATCH_CANDIDATES = 196608
STREAM4_TRIGGER_CANDIDATES = 786432
LOCAL_BEAM_WIDTH = (BEAM_WIDTH + WORLD_SIZE - 1) // WORLD_SIZE
LOGICAL_SHARD_SIZE = (LOCAL_BEAM_WIDTH + SHARD_COUNT - 1) // SHARD_COUNT
SHARD_CAPACITY_RAW = (LOGICAL_SHARD_SIZE * SHARD_CAPACITY_SCALE_PPM + 999999) // 1000000
SHARD_CAPACITY_CANDIDATES = ((SHARD_CAPACITY_RAW + STREAM4_BATCH_ALIGNMENT - 1) // STREAM4_BATCH_ALIGNMENT) * STREAM4_BATCH_ALIGNMENT
STREAM4_ACTIVE_SORT_SLOTS = 2
GLOBAL_SPILL_CAPACITY = 0
STREAM5_RECV_CAPACITY_SCALE_PPM = 2000000


In [ ]:
from pathlib import Path
import os
import shutil
import subprocess

WORK_DIR = Path('/kaggle/working')
TMP_DIR = Path('/tmp')
REPO_DIR = TMP_DIR / 'beam_solver'
CUTLASS_DIR = TMP_DIR / 'cutlass'
BUILD_DIR = TMP_DIR / 'beam_build'

def run_checked(cmd, cwd=None, env=None):
    print('+ ' + ' '.join(map(str, cmd)))
    subprocess.run(list(map(str, cmd)), cwd=cwd, env=env, check=True)

SOURCE_ARCHIVE_PATH = Path('/kaggle/input') / SOURCE_ARCHIVE_DATASET / SOURCE_ARCHIVE_NAME

for transient_dir in (REPO_DIR, BUILD_DIR):
    if transient_dir.exists():
        shutil.rmtree(transient_dir)
if SOURCE_ARCHIVE_PATH.exists():
    REPO_DIR.mkdir(parents=True, exist_ok=True)
    run_checked(['tar', '-xzf', SOURCE_ARCHIVE_PATH, '-C', REPO_DIR])
else:
    run_checked(['git', 'clone', '--branch', GITHUB_BRANCH, '--depth', '1', GITHUB_REPO_URL, REPO_DIR])

if not (CUTLASS_DIR / 'include').exists():
    if CUTLASS_DIR.exists():
        shutil.rmtree(CUTLASS_DIR)
    run_checked(['git', 'clone', '--depth', '1', 'https://github.com/NVIDIA/cutlass.git', CUTLASS_DIR])

depth_logs = 'ON' if ENABLE_DEPTH_LOGS else 'OFF'
debug_logs = 'ON' if ENABLE_DEBUG_LOGS else 'OFF'
debug_master = 'ON' if ENABLE_DEBUG else 'OFF'
debug_stream_timing = 'ON' if DEBUG_STREAM_TIMING else 'OFF'
debug_inference_trace = 'ON' if DEBUG_INFERENCE_TRACE else 'OFF'
debug_path_trace = 'ON' if DEBUG_PATH_TRACE else 'OFF'
debug_final_validate = 'ON' if DEBUG_FINAL_VALIDATE else 'OFF'
debug_final_exchange_trace = 'ON' if DEBUG_FINAL_EXCHANGE_TRACE else 'OFF'
debug_final_histogram_trace = 'ON' if DEBUG_FINAL_HISTOGRAM_TRACE else 'OFF'
debug_stream4_histogram_trace = 'ON' if DEBUG_STREAM4_HISTOGRAM_TRACE else 'OFF'
run_checked([
    'cmake', '-S', REPO_DIR, '-B', BUILD_DIR, '-GNinja',
    '-DCMAKE_BUILD_TYPE=Release',
    f'-DCUTLASS_DIR={CUTLASS_DIR}',
    f'-DBEAM_ENABLE_DEBUG={debug_master}',
    f'-DBEAM_ENABLE_DEPTH_LOGS={depth_logs}',
    f'-DBEAM_ENABLE_DEBUG_LOGS={debug_logs}',
    f'-DBEAM_DEBUG_STREAM_TIMING={debug_stream_timing}',
    f'-DBEAM_DEBUG_INFERENCE_TRACE={debug_inference_trace}',
    f'-DBEAM_DEBUG_PATH_TRACE={debug_path_trace}',
    f'-DBEAM_DEBUG_FINAL_VALIDATE={debug_final_validate}',
    f'-DBEAM_DEBUG_FINAL_EXCHANGE_TRACE={debug_final_exchange_trace}',
    f'-DBEAM_DEBUG_FINAL_HISTOGRAM_TRACE={debug_final_histogram_trace}',
    f'-DBEAM_DEBUG_STREAM4_HISTOGRAM_TRACE={debug_stream4_histogram_trace}',
])
run_checked(['cmake', '--build', BUILD_DIR, '--target', 'production_runner', '-j', '2'])


In [ ]:
import re
import sys
import threading
import time
import pandas as pd

KNOWN_SOLUTION_PATHS = {}


LOG_DIR = WORK_DIR / 'run_logs'
LOG_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_CSV = WORK_DIR / 'beam_run_results.csv'
SUBMISSION_CSV = WORK_DIR / 'submission.csv'
SOLVED_RE = re.compile(r'puzzle_solved=(\d+) puzzle_id=(\d+) seconds=([0-9.eE+-]+) solution_length=(-?\d+) solution=(.*)$')

def live_log_enabled(rank: int) -> bool:
    if LIVE_LOG_RANKS == 'all':
        return True
    if LIVE_LOG_RANKS == 'none':
        return False
    return rank in set(LIVE_LOG_RANKS)

def stream_rank_log(rank: int, proc, log_path: Path):
    with log_path.open('w', encoding='utf-8') as log:
        assert proc.stdout is not None
        for raw_line in proc.stdout:
            log.write(raw_line)
            log.flush()
            if live_log_enabled(rank):
                print(f'rank={rank} {raw_line}', end='')
                sys.stdout.flush()

def run_puzzle(puzzle_id: int, puzzle_index: int):
    env = os.environ.copy()
    env['BEAM_HISTORY_MODE'] = HISTORY_MODE
    env['BEAM_HISTORY_SLOT_COUNT'] = str(HISTORY_SLOT_COUNT)
    env['BEAM_HISTORY_WORKERS'] = str(HISTORY_WORKERS)
    env['BEAM_DEPTH_LOG_EVERY'] = str(DEPTH_LOG_EVERY)
    if DEBUG_PIPELINE_STATS:
        env['BEAM_DEBUG_PIPELINE_STATS'] = '1'
    env['BEAM_WEIGHT_DIR'] = str(REPO_DIR / 'stream1_weights')
    env['BEAM_RUNTIME_CONFIG_MODE'] = RUNTIME_CONFIG_MODE
    env['BEAM_SHARD_BUFFER_COUNT'] = str(SHARD_BUFFER_COUNT)
    env['BEAM_STREAM3_RING_SLOTS'] = str(STREAM3_RING_SLOTS)
    env['BEAM_SHARD_COUNT'] = str(SHARD_COUNT)
    env['BEAM_STREAM4_BATCH_CANDIDATES'] = str(STREAM4_BATCH_CANDIDATES)
    env['BEAM_STREAM4_TRIGGER_CANDIDATES'] = str(STREAM4_TRIGGER_CANDIDATES)
    env['BEAM_SHARD_CAPACITY_CANDIDATES'] = str(SHARD_CAPACITY_CANDIDATES)
    env['BEAM_STREAM4_ACTIVE_SORT_SLOTS'] = str(STREAM4_ACTIVE_SORT_SLOTS)
    env['BEAM_GLOBAL_SPILL_CAPACITY'] = str(GLOBAL_SPILL_CAPACITY)
    env['BEAM_STREAM5_RECV_CAPACITY_SCALE_PPM'] = str(STREAM5_RECV_CAPACITY_SCALE_PPM)
    base_cmd = [str(BUILD_DIR / 'production_runner'), str(puzzle_id), str(DEPTH_LIMIT), str(BEAM_WIDTH)]
    launch_id = f'p{puzzle_id}_{int(time.time())}'
    nccl_id_file = TMP_DIR / f'beam_solver_nccl_{launch_id}.bin'
    if nccl_id_file.exists():
        nccl_id_file.unlink()
    procs = []
    log_paths = []
    log_threads = []
    started = time.perf_counter()
    for rank in range(WORLD_SIZE):
        rank_env = env.copy()
        rank_env['WORLD_SIZE'] = str(WORLD_SIZE)
        rank_env['RANK'] = str(rank)
        rank_env['LOCAL_RANK'] = str(rank)
        rank_env['BEAM_NCCL_ID_FILE'] = str(nccl_id_file)
        log_path = LOG_DIR / f'puzzle_{puzzle_id}_rank{rank}.log'
        proc = subprocess.Popen(base_cmd, cwd=REPO_DIR, env=rank_env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        thread = threading.Thread(target=stream_rank_log, args=(rank, proc, log_path), daemon=True)
        thread.start()
        procs.append(proc)
        log_paths.append(log_path)
        log_threads.append(thread)
    timed_out = False
    deadline = started + RUN_TIMEOUT_SEC if RUN_TIMEOUT_SEC is not None and RUN_TIMEOUT_SEC > 0 else None
    while True:
        if all(proc.poll() is not None for proc in procs):
            break
        if deadline is not None and time.perf_counter() >= deadline:
            timed_out = True
            print(f'auto_stop_timeout_sec={RUN_TIMEOUT_SEC} puzzle_id={puzzle_id}')
            for proc in procs:
                if proc.poll() is None:
                    proc.terminate()
            grace_deadline = time.perf_counter() + 10.0
            while time.perf_counter() < grace_deadline and any(proc.poll() is None for proc in procs):
                time.sleep(0.2)
            for proc in procs:
                if proc.poll() is None:
                    proc.kill()
            break
        time.sleep(1.0)
    codes = [proc.wait() for proc in procs]
    for thread in log_threads:
        thread.join()
    elapsed = time.perf_counter() - started
    parsed = None
    for rank, log_path in enumerate(log_paths):
        with log_path.open('r', encoding='utf-8', errors='replace') as log:
            for raw_line in log:
                line = raw_line.rstrip('\n')
                match = SOLVED_RE.search(line)
                if match:
                    parsed = match
                    if not live_log_enabled(rank):
                        print(f'rank={rank} {line}')
                elif (not live_log_enabled(rank)) and (ENABLE_DEPTH_LOGS or line.startswith('track_solution_')):
                    print(f'rank={rank} {line}')
    failed_codes = [code for code in codes if code != 0]
    joined_logs = ';'.join(str(path) for path in log_paths)
    if timed_out:
        return {'puzzle_id': puzzle_id, 'solved': 0, 'seconds': elapsed, 'length': None, 'solution': '', 'return_code': -200, 'log_path': joined_logs}
    if failed_codes:
        result = {'puzzle_id': puzzle_id, 'solved': 0, 'seconds': elapsed, 'length': None, 'solution': '', 'return_code': max(failed_codes), 'log_path': joined_logs}
        if STOP_ON_FAILURE:
            raise RuntimeError(f'production_runner failed: puzzle_id={puzzle_id} return_codes={codes} log_paths={joined_logs}')
        return result
    if parsed is None:
        return {'puzzle_id': puzzle_id, 'solved': 0, 'seconds': elapsed, 'length': None, 'solution': '', 'return_code': max(codes), 'log_path': joined_logs}
    solved = int(parsed.group(1))
    return {
        'puzzle_id': int(parsed.group(2)),
        'solved': solved,
        'seconds': float(parsed.group(3)),
        'length': int(parsed.group(4)) if solved else None,
        'solution': parsed.group(5) if solved else '',
        'return_code': max(codes),
        'log_path': joined_logs,
    }

results = []
for index, puzzle_id in enumerate(range(START_PUZZLE_ID, START_PUZZLE_ID + PUZZLE_COUNT), start=1):
    result = run_puzzle(puzzle_id, index)
    results.append(result)
    if PUZZLE_LOG_EVERY and (index % PUZZLE_LOG_EVERY == 0):
        print(f'puzzle_progress={index}/{PUZZLE_COUNT} puzzle_id={puzzle_id} solved={result["solved"]} seconds={result["seconds"]:.6f} length={result["length"]}')
    df = pd.DataFrame(results)
    df.to_csv(RESULTS_CSV, index=False)
    solved_df = df[df['solved'] == 1][['puzzle_id', 'solution']].rename(columns={'puzzle_id': 'initial_state_id', 'solution': 'path'})
    solved_df.to_csv(SUBMISSION_CSV, index=False)

pd.DataFrame(results)


In [ ]:
from collections import Counter
import matplotlib.pyplot as plt
import pandas as pd

df = pd.read_csv(RESULTS_CSV) if RESULTS_CSV.exists() else pd.DataFrame(results)
solved = df[df['solved'] == 1].copy()
lengths = [int(x) for x in solved['length'].dropna().tolist()]
hist_path = WORK_DIR / 'solution_length_histogram.png'
if lengths:
    min_len = min(lengths)
    max_len = max(lengths)
    avg_len = sum(lengths) / len(lengths)
    mode_len = Counter(lengths).most_common(1)[0][0]
    plt.figure(figsize=(10, 5))
    plt.hist(lengths, bins=range(min_len, max_len + 2), edgecolor='black')
    plt.xlabel('solution_length')
    plt.ylabel('solved_puzzle_count')
    plt.title('Solved puzzle solution lengths')
    plt.tight_layout()
    plt.savefig(hist_path, dpi=160)
    print(f'solved_count={len(lengths)} total_count={PUZZLE_COUNT}')
    print(f'min_solution_length={min_len}')
    print(f'max_solution_length={max_len}')
    print(f'avg_solution_length={avg_len:.6f}')
    print(f'mode_solution_length={mode_len}')
    print(f'histogram_png={hist_path}')
else:
    print(f'solved_count=0 total_count={PUZZLE_COUNT}')
    print(f'histogram_png_not_created={hist_path}')

solved
